In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_path = "abfss://inputdata@storagelake9720.dfs.core.windows.net/bronze/customers"

customers_bronze = spark.read.format("delta").load("abfss://inputdata@storagelake9720.dfs.core.windows.net/bronze/customers"
)


customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"])
)

customers_silver = (
    customers_silver
    .filter("customer_id IS NOT NULL")
)


customers_silver = (
    customers_silver
    .withColumn(
        "customer_id",
        trim(col("customer_id"))
    )
    .withColumn(
        "customer_name",
        trim(col("customer_name"))
    )
    .withColumn(
        "state",
        trim(col("state"))
    )
)
customers_silver.write.mode("overwrite").format("delta").save("abfss://inputdata@storagelake9720.dfs.core.windows.net/silver/customers")


In [0]:
products_bronze = spark.read.format("delta").load("abfss://inputdata@storagelake9720.dfs.core.windows.net/bronze/products")

products_silver = (
    products_bronze
    .dropDuplicates(["product_id"])
    .filter("product_id IS NOT NULL")
    .filter("price IS NOT NULL")
    .filter("price > 0")
)
products_silver = (
    products_silver
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
)

products_silver.write.mode("overwrite").format("delta").save("abfss://inputdata@storagelake9720.dfs.core.windows.net/silver/products")

In [0]:
orders_bronze=(
    spark.read.format("delta").load("abfss://inputdata@storagelake9720.dfs.core.windows.net/bronze/orders")
)

orders_silver = (
    orders_bronze
    .dropDuplicates(["order_id"])
    .filter("order_id IS NOT NULL")
    .filter("customer_id IS NOT NULL")
    .filter("product_id IS NOT NULL")
    .filter("quantity IS NOT NULL")
    .filter("quantity > 0")
)

orders_silver = orders_silver.withColumn(
    "order_date",
    try_to_date(col("order_date"), "yyyy-MM-dd"))

orders_silver = orders_silver.filter(
    col("order_date").isNotNull())

orders_silver.write.mode("overwrite").format("delta").save("abfss://inputdata@storagelake9720.dfs.core.windows.net/silver/orders")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.customers
USING DELTA LOCATION 'abfss://inputdata@storagelake9720.dfs.core.windows.net/silver/customers';

CREATE TABLE IF NOT EXISTS ecommerce.silver.orders
USING DELTA LOCATION 'abfss://inputdata@storagelake9720.dfs.core.windows.net/silver/orders';

CREATE TABLE IF NOT EXISTS ecommerce.silver.products
USING DELTA LOCATION 'abfss://inputdata@storagelake9720.dfs.core.windows.net/silver/products';
